# Plot differentially accessible regions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.pyplot import rcParams
from adjustText import adjust_text

import project_style_py as scp
scp.load_project_palettes('/lustre/scratch124/humgen/projects_v2/cardinal_analysis/analysis/od5/annotation/freeze3_stats/freeze3_palettes.yaml'
)
scp.load_project_themes(
    '/lustre/scratch124/humgen/projects_v2/cardinal_analysis/analysis/od5/annotation/freeze3_stats/freeze3_themes_py.yaml'
)

%config InlineBackend.figure_format = 'retina'

KeyboardInterrupt: 

In [ ]:
# Set Helvetica as default font (fallback to Arial / sans-serif if not installed)

import matplotlib as mpl

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
mpl.rcParams["pdf.fonttype"] = 42   # editable text in Illustrator
mpl.rcParams["ps.fonttype"] = 42

In [ ]:
work_dir = '/lustre/scratch125/casm/teams/team215/users/od5/projects/celltherapy/ipsc/multiome/scenicplus/tf_pilot/batch1/ipsc/'

p_output = work_dir + 'DARs_archr/'

In [ ]:
dars = pd.read_csv(p_output + 'archr_DARs_perturbation_df_pos_neg.txt', sep='\t')

In [ ]:
dars

In [ ]:
grn_comb_high_conf = pd.read_csv((work_dir + 'eregulon_metadata_full_20runs_combined_min5recall.csv'), index_col=0)

In [ ]:
grn_comb_high_conf

In [ ]:
degs = pd.read_csv('/nfs/casm/team215mg/users/od5/projects/celltherapy/ipsc/multiome/tf_pilot/002_integrate_and_annotate/batch1_ipsc/ipsc_multiome_tf_degs_wilcoxon.csv')

In [ ]:
degs

In [ ]:
# Merges DAR peaks with high-confidence GRN triplets by genomic region.
# Keeps all DAR rows that have matching GRN regions.

dar_grn = (
    dars
    .merge(
        grn_comb_high_conf,
        left_on="peak",
        right_on="Region",
        how="left"
    )
)

dar_grn.head()

In [ ]:
# Merge DAR-GRN links with DEGs by matching Cell_type/group and Gene/names.

degs_renamed = degs.rename(columns={
    "group": "Cell_type",
    "names": "Gene",
    "logfoldchanges": "deg_logFC",
    "pvals_adj": "deg_FDR",
    "scores": "deg_score",
    "pvals": "deg_pval"
})

dar_grn_deg = dar_grn.merge(
    degs_renamed,
    on=["Cell_type", "Gene"],
    how="left"
)

dar_grn_deg.head()

In [ ]:
# Plot DAR volcano.
# Only significant DARs are coloured by DEG direction:
#   DAR significant + DEG up   = dark red
#   DAR significant + DEG down = dark blue
#   everything else            = grey
# Dot size increases with positive DEG logFC.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text

cell_types = ["RUNX2"]
top_n = 6

dar_fdr_cutoff = 0.05
deg_fdr_cutoff = 0.05

plot_df = dar_grn_deg[dar_grn_deg["Cell_type"].isin(cell_types)].copy()

plot_df["neg_log10_FDR"] = -np.log10(plot_df["FDR"].clip(lower=1e-300))
plot_df["neg_log10_deg_FDR"] = -np.log10(plot_df["deg_FDR"].clip(lower=1e-300))

plot_df["point_colour"] = "grey"

plot_df.loc[
    (plot_df["FDR"] < dar_fdr_cutoff) &
    (plot_df["deg_FDR"] < deg_fdr_cutoff) &
    (plot_df["deg_logFC"] > 0),
    "point_colour"
] = "darkred"

plot_df.loc[
    (plot_df["FDR"] < dar_fdr_cutoff) &
    (plot_df["deg_FDR"] < deg_fdr_cutoff) &
    (plot_df["deg_logFC"] < 0),
    "point_colour"
] = "darkblue"

plot_df["point_size"] = 10 + 2 * plot_df["neg_log10_deg_FDR"].clip(lower=0).fillna(0)

fig, ax = plt.subplots(figsize=(7, 5))

# Plot grey background first
for colour, label in [
    ("grey", "not significantly DE"),
    ("darkred", "DE up"),
    ("darkblue", "DE down")
]:
    subdf = plot_df[plot_df["point_colour"] == colour]

    ax.scatter(
        subdf["Log2FC"],
        subdf["neg_log10_FDR"],
        s=subdf["point_size"],
        c=colour,
        alpha=0.5,
        label=label,
        edgecolors="none"
    )

label_df = (
    plot_df
    .loc[plot_df["Gene"].notna()]
    .sort_values(
        ["neg_log10_FDR"],
        ascending=[False]
    )
    .drop_duplicates("peak")
    .head(top_n)
)

texts = []

for _, row in label_df.iterrows():

    label = f"{row['peak']}\n{row['Gene']}"

    texts.append(
        ax.text(
            row["Log2FC"],
            row["neg_log10_FDR"],
            label,
            fontsize=8
        )
    )

adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", lw=0.5)
)

ax.axhline(
    -np.log10(dar_fdr_cutoff),
    linestyle="--",
    color="lightgrey",
    linewidth=1
)

ax.set_xlabel("DAR Log2FC")
ax.set_ylabel("-log10(DAR FDR)")
ax.set_title("RUNX2\nDifferential accessibility and expression")

ax.legend(
    title="",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig("RUNX2_DARs_DEGs_volcano.pdf")

plt.show()

In [ ]:
dar_grn_deg.to_csv('DAR_DEG_GRN.csv')

In [ ]:
# COL1A1 entries in the original dataframe
dar_grn_deg.loc[
    dar_grn_deg['Gene'] == 'COL1A1'
].head(20)

In [ ]:
sdds